In [1]:
import os
import json
import re
import html as htmlmod
from pathlib import Path
from tqdm import tqdm
import multiprocessing as mp

import trafilatura
import nltk
from langdetect import detect, LangDetectException, DetectorFactory
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# One-time downloads (safe to re-run; it will skip if already present)
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("punkt")
nltk.download("omw-1.4")

DetectorFactory.seed = 0  # more reproducible langdetect results

STOP_WORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()

[nltk_data] Downloading package stopwords to /home/irdin/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/irdin/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/irdin/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/irdin/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [2]:
# You can run this from inside the dataset root OR point to it here.
DATASET_ROOT = Path(".")  # change to Path("/path/to/2026-01-27_201419_domain_with_snapshots") if needed

METADATA_DIR = DATASET_ROOT / "metadata"
HTMLS_DIR = DATASET_ROOT / "htmls"  # not strictly needed, but good for checks

# For testing:
N_FILES = 1000   # first 1000 metadata shard files; set to None to process all
LIMIT_RECORDS = None  # e.g., 50_000 for a quick test; None = all records in selected shards

OUTPUT_JSONL = DATASET_ROOT / "preprocessed_from_disk.jsonl"

In [3]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+")
MENTION_HASH_RE = re.compile(r"@\S+|#\S+")
NUMBER_RE = re.compile(r"\b\d+\b")
PUNCT_RE = re.compile(r"[^\w\s]")
REPEAT_RE = re.compile(r"(.)\1{2,}")
WS_RE = re.compile(r"\s+")

def detect_english(text: str) -> bool:
    if not isinstance(text, str) or not text.strip():
        return False
    try:
        return detect(text) == "en"
    except LangDetectException:
        return False

def preprocess_text(text: str) -> str:
    """
    Similar to your CSV version: clean -> tokenize -> lemmatize -> remove stopwords -> length filter
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    try:
        text = URL_RE.sub("", text)
        text = EMAIL_RE.sub("", text)
        text = MENTION_HASH_RE.sub("", text)
        text = NUMBER_RE.sub("", text)
        text = PUNCT_RE.sub(" ", text)
        text = REPEAT_RE.sub(r"\1\1", text.lower())
        text = WS_RE.sub(" ", text).strip()

        tokens = word_tokenize(text)
        lemmas = [
            LEMMATIZER.lemmatize(tok)
            for tok in tokens
            if tok not in STOP_WORDS and 2 < len(tok) <= 25
        ]
        return " ".join(lemmas)
    except Exception:
        return ""

In [4]:
def iter_metadata_shards(metadata_dir: Path, n_files=None):
    shards = sorted(metadata_dir.glob("*.jsonl"))
    if n_files is not None:
        shards = shards[:n_files]
    return shards

In [5]:
def _worker_process_record(rec: dict, dataset_root_str: str):
    """
    Worker receives a metadata record dict, reads html_file_location from disk,
    runs trafilatura + langdetect + preprocess, returns JSON string.
    """
    dataset_root = Path(dataset_root_str)

    rel = rec.get("html_file_location")
    if not rel:
        rec["trafilatura_text"] = ""
        rec["is_english_trafilatura"] = False
        rec["preprocessed_trafilatura"] = ""
        rec["trafilatura_status"] = "missing_html_file_location"
        return json.dumps(rec, ensure_ascii=False)

    html_path = dataset_root / rel
    if not html_path.exists():
        rec["trafilatura_text"] = ""
        rec["is_english_trafilatura"] = False
        rec["preprocessed_trafilatura"] = ""
        rec["trafilatura_status"] = "html_file_missing"
        return json.dumps(rec, ensure_ascii=False)

    try:
        content = html_path.read_text(encoding="utf-8", errors="replace")
    except Exception:
        rec["trafilatura_text"] = ""
        rec["is_english_trafilatura"] = False
        rec["preprocessed_trafilatura"] = ""
        rec["trafilatura_status"] = "html_read_error"
        return json.dumps(rec, ensure_ascii=False)

    extracted = trafilatura.extract(content)
    if not extracted:
        rec["trafilatura_text"] = ""
        rec["is_english_trafilatura"] = False
        rec["preprocessed_trafilatura"] = ""
        rec["trafilatura_status"] = "trafilatura_empty"
        return json.dumps(rec, ensure_ascii=False)

    extracted = htmlmod.unescape(extracted)
    rec["trafilatura_text"] = extracted

    is_eng = detect_english(extracted)
    rec["is_english_trafilatura"] = is_eng

    if is_eng:
        rec["preprocessed_trafilatura"] = preprocess_text(extracted)
        rec["trafilatura_status"] = "ok_en"
    else:
        rec["preprocessed_trafilatura"] = ""
        rec["trafilatura_status"] = "ok_non_en"

    return json.dumps(rec, ensure_ascii=False)

In [6]:
from functools import partial

def process_dataset_to_jsonl(
    dataset_root: Path,
    metadata_dir: Path,
    out_jsonl: Path,
    n_files=None,
    limit_records=None,
    workers=None,
    chunk_size=100
):
    if not metadata_dir.exists():
        raise RuntimeError(f"metadata dir not found: {metadata_dir}")
    if workers is None:
        workers = max(1, mp.cpu_count() - 1)

    shards = iter_metadata_shards(metadata_dir, n_files=n_files)
    if not shards:
        raise RuntimeError("No metadata shard files found.")

    out_jsonl.parent.mkdir(parents=True, exist_ok=True)

    written = 0
    bad_json = 0

    worker_fn = partial(_worker_process_record, dataset_root_str=str(dataset_root))

    with mp.Pool(processes=workers) as pool, out_jsonl.open("w", encoding="utf-8") as fout:
        print(f"Shards: {len(shards)} | Workers: {workers} | chunk_size: {chunk_size}")
        for shard in shards:
            print(f"Processing shard: {shard.name}")
            with shard.open("r", encoding="utf-8", errors="replace") as fin:
                def records_iter():
                    nonlocal bad_json, written
                    for line in fin:
                        if limit_records is not None and written >= limit_records:
                            return
                        line = line.strip()
                        if not line:
                            continue
                        try:
                            rec = json.loads(line)
                        except json.JSONDecodeError:
                            bad_json += 1
                            continue
                        yield rec

                for out_line in tqdm(
                    pool.imap(worker_fn, records_iter(), chunksize=chunk_size),
                    desc=f"Shard {shard.name}",
                    leave=False
                ):
                    fout.write(out_line + "\n")
                    written += 1
                    if limit_records is not None and written >= limit_records:
                        break

            if limit_records is not None and written >= limit_records:
                break

    print("\nDone.")
    print(f"Wrote records: {written:,}")
    print(f"Bad JSON lines skipped: {bad_json:,}")
    print(f"Output: {out_jsonl.resolve()}")

In [7]:
process_dataset_to_jsonl(
    dataset_root=DATASET_ROOT,
    metadata_dir=METADATA_DIR,
    out_jsonl=OUTPUT_JSONL,
    n_files=N_FILES,          # 1000 for test; None for all
    limit_records=LIMIT_RECORDS,
    workers=8,                # adjust to your CPU
    chunk_size=100            # 50–200 is typical for heavy work like trafilatura+langdetect
)

Shards: 256 | Workers: 8 | chunk_size: 100
Processing shard: 00.jsonl


Processing shard: 01.jsonl


Processing shard: 02.jsonl


Processing shard: 03.jsonl


Processing shard: 04.jsonl


...

Processing shard: fc.jsonl


Processing shard: fd.jsonl


Processing shard: fe.jsonl


Processing shard: ff.jsonl



Done.
Wrote records: 11,403,638
Bad JSON lines skipped: 0
Output: /home/darknet/2026-01-27_201419_domain_with_snapshots/preprocessed_from_disk.jsonl


In [8]:
from itertools import islice

def preview_jsonl(path, n=5, max_len=300):
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(islice(f, n), 1):
            rec = json.loads(line)
            print(f"\n--- Entry {i} ---")
            for k, v in rec.items():
                if isinstance(v, str) and len(v) > max_len:
                    print(f"{k}: {v[:max_len]}... (len={len(v)})")
                else:
                    print(f"{k}: {v}")

preview_jsonl(OUTPUT_JSONL, n=5)


--- Entry 1 ---
created_at: 1585293617000
darknet_type: torv2
domain_id: 163884
domain_url: http://hack3q3l3szn2aib.onion
file_size: 10104
html_file_location: htmls/00/8c/008c6311d24df8c8e190c79e3d43874e.html
page_id: 808271
path: /
path_hash: 42099b4af021e53fd8fd4e056c2568d7c2e3ffa8
snapshot_hash: 008c6311d24df8c8e190c79e3d43874e
snapshot_id: 24417808
tags: Cybercrime,English,Hacking,Service Provider,Social Engineering,Social Media
title: Hacker | Cyber Crime Solution
trafilatura_text: Do you want to find out if your website, computer or network can be or has been hacked?
Would you like to hack into a computer, website or network?
Social Media Threats
Has your Facebook, Twitter or Google+ account been hacked? We can help get it restored and track the person who did it in many case... (len=2713)
is_english_trafilatura: True
preprocessed_trafilatura: want find website computer network hacked would like hack computer website network social medium threat facebook twitter google account h